# 外食データの取得・価格

担当：王


In [ ]:
import csv
import json
import time
from pathlib import Path
from urllib.request import Request, urlopen

BASE_URL = "https://tabecal.com/api/v1/items"
LIMIT = 50
OUT_DIR = Path(__file__).parent

CSV_PATH = OUT_DIR / "tabecal_items_with_price.csv"
JSON_PATH = OUT_DIR / "tabecal_items_with_price.json"
PROGRESS_PATH = OUT_DIR / "tabecal_progress.json"


def fetch_page(offset):
    url = f"{BASE_URL}?has=price&limit={LIMIT}&offset={offset}"
    request = Request(
        url,
        headers={"User-Agent": "StudentResearch/1.0"}
    )

    with urlopen(request, timeout=30) as response:
        return json.loads(response.read().decode("utf-8"))


def extract_items(result):
    if isinstance(result, list):
        return result
    if isinstance(result, dict):
        for key in ("items", "data"):
            if isinstance(result.get(key), list):
                return result[key]
    return []


# 读取之前已保存的进度
if PROGRESS_PATH.exists():
    with open(PROGRESS_PATH, "r", encoding="utf-8") as f:
        items = json.load(f)
    print(f"读取已有进度：{len(items)} 条")
else:
    items = []

# 根据已有记录确定下一页
offset = len(items)

while True:
    try:
        print(f"正在获取 offset={offset} 的数据……")
        result = fetch_page(offset)
        batch = extract_items(result)

        if not batch:
            print("没有更多数据，结束提取。")
            break

        # 只保留有价格的记录
        batch = [
            item for item in batch
            if isinstance(item, dict)
            and item.get("price") is not None
        ]

        items.extend(batch)
        offset += LIMIT

        # 每批都保存进度
        with open(PROGRESS_PATH, "w", encoding="utf-8") as f:
            json.dump(items, f, ensure_ascii=False, indent=2)

        print(f"当前已保存：{len(items)} 条")
        time.sleep(0.3)

    except Exception as e:
        print(f"本次请求失败：{e}")
        print("已保存的进度还在。请检查网络后再运行。")
        break

# 保存最终 JSON
with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

# 保存最终 CSV
if items:
    columns = list(dict.fromkeys(
        key for item in items for key in item.keys()
    ))

    with open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(
            f, fieldnames=columns, extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(items)

print("\n提取结束")
print(f"记录总数：{len(items)}")
print(f"CSV：{CSV_PATH}")
print(f"JSON：{JSON_PATH}")


In [ ]:

import json
from pathlib import Path
from collections import defaultdict

folder = Path(__file__).parent
json_path = folder / "tabecal_items_with_price.json"

if not json_path.exists():
    json_path = folder / "tabecal_progress.json"

if not json_path.exists():
    print("没有找到 JSON 文件，请检查 data 文件夹。")
    raise SystemExit

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

if isinstance(data, dict):
    data = data.get("items", data.get("data", []))

targets = [
    "すき家", "吉野家", "松屋", "なか卯", "やよい軒",
    "ガスト", "サイゼリヤ", "丸亀製麺",
    "CoCo壱番屋", "マクドナルド"
]

# 按品牌统计有价格的菜单
menus = defaultdict(list)

for item in data:
    chain = str(item.get("chain", "")).strip()

    if chain in targets and item.get("price") is not None:
        menus[chain].append(item)

print("===== 目标店铺的菜单数据 =====")

for chain in targets:
    records = menus[chain]
    print(f"\n【{chain}】有价格的菜单：{len(records)} 条")

    for item in records[:10]:
        print(
            f"  菜名：{item.get('name', '未知')} | "
            f"价格：{item.get('price')}日元 | "
            f"热量：{item.get('kcal', '未提供')} kcal"
        )

# 把匹配到的全部菜单保存为新 CSV
import csv

output = folder / "selected_restaurant_menus.csv"
selected = [
    item
    for chain in targets
    for item in menus[chain]
]

if selected:
    columns = list(dict.fromkeys(
        key for item in selected for key in item.keys()
    ))

    with open(output, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(selected)

    print(f"\n已保存：{output}")
    print(f"匹配到的菜单总数：{len(selected)}")
else:
    print("\n当前文件没有匹配到目标店铺的带价格菜单。")
    print("请检查品牌名称或确认 JSON 是否包含完整数据。")
